# Multi-Timeframe LSTM Training

This notebook implements a **multi-timeframe approach** to overcome the 5000 candle API limitation.

## Strategy
- Extract data from **M15, H1, and H4** timeframes
- Align timeframes so each H4 candle has corresponding M15 and H1 context
- Calculate technical indicators for each timeframe
- Train LSTM with **multi-timeframe features** for better predictions

## Expected Benefits
- More training data (combine 3 timeframes instead of 1)
- Better context (short-term M15 + medium-term H1 + long-term H4)
- Improved directional accuracy (target: >55%)

**Run all cells in order: Cell → Run All**

In [ ]:
# Cell 1: Imports and Configuration
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configuration
SYMBOL = 'Volatility 100 Index'
TIMEFRAMES = ['M15', 'H1', 'H4']  # Multiple timeframes
TARGET_TIMEFRAME = 'H4'  # Primary timeframe for predictions

# LSTM Hyperparameters
SEQUENCE_LENGTH = 60  # Number of timesteps to look back
PREDICTION_HORIZON = 1  # Predict next timestep
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 100
PATIENCE = 15  # Early stopping patience

print("\n" + "="*60)
print("Multi-Timeframe LSTM Configuration")
print("="*60)
print(f"Symbol: {SYMBOL}")
print(f"Timeframes: {', '.join(TIMEFRAMES)}")
print(f"Target Timeframe: {TARGET_TIMEFRAME}")
print(f"Sequence Length: {SEQUENCE_LENGTH}")
print(f"Hidden Size: {HIDDEN_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Max Epochs: {EPOCHS}")
print("="*60)

In [ ]:
# Cell 2: Load Multi-Timeframe Data
from src.utils.helpers import load_data

def load_multi_timeframe_data(symbol, timeframes):
    """Load data from multiple timeframes and align them"""
    data_dict = {}
    
    for tf in timeframes:
        df = load_data(symbol, tf)
        if df is None:
            print(f"❌ Failed to load {symbol} {tf}")
            print(f"   Please extract data first:")
            print(f"   python src/extractors/deriv_api_extractor.py --symbol '{symbol}' --timeframe {tf} --days 365")
            return None
        data_dict[tf] = df
        print(f"✅ Loaded {symbol} {tf}: {len(df)} candles")
    
    return data_dict

print("Loading multi-timeframe data...\n")
data_dict = load_multi_timeframe_data(SYMBOL, TIMEFRAMES)

if data_dict:
    print("\n" + "="*60)
    print("Data Successfully Loaded")
    print("="*60)
    for tf, df in data_dict.items():
        print(f"{tf}: {len(df)} candles from {df.index[0]} to {df.index[-1]}")
    print("="*60)

In [ ]:
# Cell 3: Calculate Technical Indicators for Each Timeframe
def calculate_indicators(df, prefix=''):
    """Calculate technical indicators with optional prefix for column names"""
    df = df.copy()
    
    # Returns
    df[f'{prefix}Returns'] = df['Close'].pct_change()
    
    # Simple Moving Averages
    df[f'{prefix}SMA_10'] = df['Close'].rolling(window=10).mean()
    df[f'{prefix}SMA_20'] = df['Close'].rolling(window=20).mean()
    df[f'{prefix}SMA_50'] = df['Close'].rolling(window=50).mean()
    
    # Exponential Moving Averages
    df[f'{prefix}EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df[f'{prefix}EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    
    # RSI (Relative Strength Index)
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df[f'{prefix}RSI'] = 100 - (100 / (1 + rs))
    
    # MACD
    exp1 = df['Close'].ewm(span=12, adjust=False).mean()
    exp2 = df['Close'].ewm(span=26, adjust=False).mean()
    df[f'{prefix}MACD'] = exp1 - exp2
    df[f'{prefix}MACD_signal'] = df[f'{prefix}MACD'].ewm(span=9, adjust=False).mean()
    
    # Bollinger Bands
    df[f'{prefix}BB_middle'] = df['Close'].rolling(window=20).mean()
    bb_std = df['Close'].rolling(window=20).std()
    df[f'{prefix}BB_upper'] = df[f'{prefix}BB_middle'] + (bb_std * 2)
    df[f'{prefix}BB_lower'] = df[f'{prefix}BB_middle'] - (bb_std * 2)
    df[f'{prefix}BB_width'] = df[f'{prefix}BB_upper'] - df[f'{prefix}BB_lower']
    
    # ATR (Average True Range)
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    df[f'{prefix}ATR'] = true_range.rolling(14).mean()
    
    # Volatility
    df[f'{prefix}Volatility'] = df['Close'].rolling(window=20).std()
    
    # Volume-related features (if volume exists)
    if 'Volume' in df.columns:
        df[f'{prefix}Volume_SMA'] = df['Volume'].rolling(window=20).mean()
    
    return df

print("Calculating technical indicators for each timeframe...\n")

# Calculate indicators for each timeframe
for tf in TIMEFRAMES:
    prefix = f'{tf}_'
    data_dict[tf] = calculate_indicators(data_dict[tf], prefix=prefix)
    print(f"✅ {tf}: Added {len([col for col in data_dict[tf].columns if col.startswith(prefix)])} indicators")

print("\n" + "="*60)
print("Technical Indicators Calculated")
print("="*60)

In [ ]:
# Cell 4: Align Multi-Timeframe Data
def align_multi_timeframe(data_dict, target_timeframe):
    """
    Align multiple timeframes to the target timeframe.
    For each target timeframe candle, get corresponding features from higher frequency timeframes.
    """
    # Use target timeframe as base
    df_target = data_dict[target_timeframe].copy()
    
    # Remove prefix from target timeframe columns
    target_prefix = f'{target_timeframe}_'
    df_target.columns = [col.replace(target_prefix, '') for col in df_target.columns]
    
    # Select features from target timeframe
    target_features = ['Close', 'Returns', 'SMA_10', 'SMA_20', 'EMA_10', 
                      'RSI', 'MACD', 'BB_middle', 'BB_width', 'ATR', 'Volatility']
    
    # Keep only existing columns
    target_features = [f for f in target_features if f in df_target.columns]
    df_aligned = df_target[target_features].copy()
    
    # For each higher frequency timeframe, add features using asof merge
    for tf in data_dict.keys():
        if tf == target_timeframe:
            continue
        
        df_tf = data_dict[tf].copy()
        
        # Select most important features from this timeframe
        tf_prefix = f'{tf}_'
        selected_features = [
            f'{tf_prefix}Close', f'{tf_prefix}Returns', f'{tf_prefix}SMA_10',
            f'{tf_prefix}RSI', f'{tf_prefix}MACD', f'{tf_prefix}Volatility'
        ]
        
        # Keep only existing columns
        selected_features = [f for f in selected_features if f in df_tf.columns]
        df_tf_selected = df_tf[selected_features]
        
        # Merge using asof (forward fill to match timestamps)
        df_aligned = pd.merge_asof(
            df_aligned.sort_index(),
            df_tf_selected.sort_index(),
            left_index=True,
            right_index=True,
            direction='backward'
        )
    
    # Drop rows with NaN values
    df_aligned = df_aligned.dropna()
    
    print(f"\n✅ Aligned dataset: {len(df_aligned)} rows × {len(df_aligned.columns)} features")
    print(f"   Timeframe range: {df_aligned.index[0]} to {df_aligned.index[-1]}")
    
    return df_aligned

print("Aligning multi-timeframe data...")
df_multi = align_multi_timeframe(data_dict, TARGET_TIMEFRAME)

print("\n" + "="*60)
print("Multi-Timeframe Features")
print("="*60)
print(f"Total features: {len(df_multi.columns)}")
print(f"Total samples: {len(df_multi)}")
print("\nFeature breakdown:")
for tf in TIMEFRAMES:
    tf_cols = [col for col in df_multi.columns if col.startswith(f'{tf}_')]
    print(f"  {tf}: {len(tf_cols)} features")
base_cols = [col for col in df_multi.columns if not any(col.startswith(f'{tf}_') for tf in TIMEFRAMES)]
print(f"  {TARGET_TIMEFRAME} (target): {len(base_cols)} features")
print("="*60)

# Display first few rows
print("\nFirst few rows:")
print(df_multi.head())

In [ ]:
# Cell 5: Prepare Sequences for LSTM
class TimeSeriesDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = torch.FloatTensor(sequences)
        self.targets = torch.FloatTensor(targets)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

def prepare_sequences(data, sequence_length=60, prediction_horizon=1, train_ratio=0.7, val_ratio=0.15):
    """
    Prepare sequences for LSTM training with multi-timeframe features
    """
    # Normalize features
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    
    # Create sequences
    sequences = []
    targets = []
    
    for i in range(len(data_scaled) - sequence_length - prediction_horizon + 1):
        # Sequence: past sequence_length timesteps with all features
        seq = data_scaled[i:i + sequence_length]
        
        # Target: next Close price (first column)
        target_idx = i + sequence_length + prediction_horizon - 1
        target = data_scaled[target_idx, 0]  # Close price is first column
        
        sequences.append(seq)
        targets.append(target)
    
    sequences = np.array(sequences)
    targets = np.array(targets)
    
    # Split into train, validation, test
    train_size = int(len(sequences) * train_ratio)
    val_size = int(len(sequences) * val_ratio)
    
    X_train = sequences[:train_size]
    y_train = targets[:train_size]
    
    X_val = sequences[train_size:train_size + val_size]
    y_val = targets[train_size:train_size + val_size]
    
    X_test = sequences[train_size + val_size:]
    y_test = targets[train_size + val_size:]
    
    # Create datasets
    train_dataset = TimeSeriesDataset(X_train, y_train)
    val_dataset = TimeSeriesDataset(X_val, y_val)
    test_dataset = TimeSeriesDataset(X_test, y_test)
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    return train_loader, val_loader, test_loader, scaler

print("Preparing sequences for LSTM training...\n")

train_loader, val_loader, test_loader, scaler = prepare_sequences(
    df_multi.values,
    sequence_length=SEQUENCE_LENGTH,
    prediction_horizon=PREDICTION_HORIZON
)

print("="*60)
print("Dataset Split")
print("="*60)
print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Total sequences: {len(train_loader.dataset) + len(val_loader.dataset) + len(test_loader.dataset)}")
print("="*60)

# Show sample batch shape
sample_batch = next(iter(train_loader))
print(f"\nSample batch shape: {sample_batch[0].shape}")
print(f"  (batch_size, sequence_length, num_features)")
print(f"  ({sample_batch[0].shape[0]}, {sample_batch[0].shape[1]}, {sample_batch[0].shape[2]})")

In [ ]:
# Cell 6: Define Multi-Timeframe LSTM Model
class MultiTimeframeLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(MultiTimeframeLSTM, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1)
        )
    
    def forward(self, x):
        # LSTM forward
        lstm_out, _ = self.lstm(x)
        
        # Take output from last timestep
        last_output = lstm_out[:, -1, :]
        
        # Fully connected layers
        output = self.fc(last_output)
        
        return output.squeeze()

# Initialize model
input_size = df_multi.shape[1]  # Number of features
model = MultiTimeframeLSTM(
    input_size=input_size,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("="*60)
print("Multi-Timeframe LSTM Model")
print("="*60)
print(f"Input size (features): {input_size}")
print(f"Hidden size: {HIDDEN_SIZE}")
print(f"Number of layers: {NUM_LAYERS}")
print(f"Dropout: {DROPOUT}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("="*60)
print(f"\nModel architecture:")
print(model)

In [ ]:
# Cell 7: Training Loop with Early Stopping
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100, patience=15):
    """
    Train the LSTM model with early stopping
    """
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    print("\nStarting training...\n")
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            # Forward pass
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        # Print progress every 5 epochs
        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping triggered at epoch {epoch+1}")
                print(f"Best validation loss: {best_val_loss:.6f}")
                break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return train_losses, val_losses

# Train the model
train_losses, val_losses = train_model(
    model, train_loader, val_loader, criterion, optimizer,
    epochs=EPOCHS, patience=PATIENCE
)

print("\n" + "="*60)
print("Training Complete")
print("="*60)

In [ ]:
# Cell 8: Visualize Training History
plt.figure(figsize=(12, 5))

plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Multi-Timeframe LSTM Training History', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final Training Loss: {train_losses[-1]:.6f}")
print(f"Final Validation Loss: {val_losses[-1]:.6f}")
print(f"Best Validation Loss: {min(val_losses):.6f}")

In [ ]:
# Cell 9: Evaluate on Test Set
def evaluate_model(model, test_loader, scaler):
    """
    Evaluate the model on test set and calculate metrics
    """
    model.eval()
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            
            outputs = model(X_batch)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(y_batch.numpy())
    
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    
    # Inverse transform to get actual prices
    # Create dummy array with same shape as original features
    n_features = scaler.n_features_in_
    dummy_pred = np.zeros((len(predictions), n_features))
    dummy_actual = np.zeros((len(actuals), n_features))
    
    dummy_pred[:, 0] = predictions  # Close price is first column
    dummy_actual[:, 0] = actuals
    
    predictions_inv = scaler.inverse_transform(dummy_pred)[:, 0]
    actuals_inv = scaler.inverse_transform(dummy_actual)[:, 0]
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(actuals_inv, predictions_inv))
    mae = mean_absolute_error(actuals_inv, predictions_inv)
    
    # Calculate directional accuracy
    actual_direction = np.diff(actuals_inv) > 0
    pred_direction = np.diff(predictions_inv) > 0
    directional_accuracy = np.mean(actual_direction == pred_direction) * 100
    
    return {
        'predictions': predictions_inv,
        'actuals': actuals_inv,
        'rmse': rmse,
        'mae': mae,
        'directional_accuracy': directional_accuracy
    }

print("Evaluating model on test set...\n")
results = evaluate_model(model, test_loader, scaler)

print("="*60)
print("Test Set Results")
print("="*60)
print(f"RMSE: {results['rmse']:.2f}")
print(f"MAE: {results['mae']:.2f}")
print(f"Directional Accuracy: {results['directional_accuracy']:.2f}%")
print("="*60)

if results['directional_accuracy'] > 55:
    print("\n✅ EXCELLENT! Model beats random predictions (>55%)")
    print("   This model can potentially be used for trading!")
elif results['directional_accuracy'] > 50:
    print("\n⚠️  Model slightly better than random (50-55%)")
    print("   Needs improvement before trading")
else:
    print("\n❌ Model performs worse than random (<50%)")
    print("   Needs significant improvement")

In [ ]:
# Cell 10: Visualize Predictions vs Actuals
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Plot 1: Time series comparison
plot_range = min(200, len(results['actuals']))  # Show last 200 points
axes[0].plot(results['actuals'][-plot_range:], label='Actual', linewidth=2, alpha=0.7)
axes[0].plot(results['predictions'][-plot_range:], label='Predicted', linewidth=2, alpha=0.7)
axes[0].set_xlabel('Time Step', fontsize=12)
axes[0].set_ylabel('Close Price', fontsize=12)
axes[0].set_title('Multi-Timeframe LSTM: Predictions vs Actual (Last 200 points)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Scatter plot
axes[1].scatter(results['actuals'], results['predictions'], alpha=0.5, s=20)
min_val = min(results['actuals'].min(), results['predictions'].min())
max_val = max(results['actuals'].max(), results['predictions'].max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Price', fontsize=12)
axes[1].set_ylabel('Predicted Price', fontsize=12)
axes[1].set_title('Prediction Accuracy Scatter Plot', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Predict Next Price
def predict_next(model, data, scaler, sequence_length):
    """
    Predict the next price using the most recent data
    """
    model.eval()
    
    # Get last sequence
    last_sequence = data[-sequence_length:]
    last_sequence_scaled = scaler.transform(last_sequence)
    
    # Convert to tensor
    X = torch.FloatTensor(last_sequence_scaled).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        prediction_scaled = model(X).cpu().numpy()[0]
    
    # Inverse transform
    n_features = scaler.n_features_in_
    dummy = np.zeros((1, n_features))
    dummy[0, 0] = prediction_scaled
    prediction = scaler.inverse_transform(dummy)[0, 0]
    
    return prediction

# Predict next price
current_price = df_multi['Close'].iloc[-1]
next_price = predict_next(model, df_multi.values, scaler, SEQUENCE_LENGTH)

price_change = next_price - current_price
price_change_pct = (price_change / current_price) * 100

print("="*60)
print("Next Price Prediction")
print("="*60)
print(f"Current Price: {current_price:.2f}")
print(f"Predicted Next Price ({TARGET_TIMEFRAME}): {next_price:.2f}")
print(f"Expected Change: {price_change:+.2f} ({price_change_pct:+.2f}%)")

if price_change > 0:
    print(f"\n📈 Signal: BUY (Expecting price increase)")
else:
    print(f"\n📉 Signal: SELL (Expecting price decrease)")

print("="*60)
print("⚠️  WARNING: This is for educational purposes only!")
print("   Always backtest before trading real money.")
print("="*60)

In [ ]:
# Cell 12: Comparison Summary and Recommendations
print("="*80)
print(" " * 20 + "MULTI-TIMEFRAME LSTM SUMMARY")
print("="*80)

print("\n📊 DATA CONFIGURATION:")
print(f"   Symbol: {SYMBOL}")
print(f"   Timeframes used: {', '.join(TIMEFRAMES)}")
print(f"   Target timeframe: {TARGET_TIMEFRAME}")
print(f"   Total features: {df_multi.shape[1]}")
print(f"   Training samples: {len(train_loader.dataset)}")
print(f"   Test samples: {len(test_loader.dataset)}")

print("\n🧠 MODEL CONFIGURATION:")
print(f"   Architecture: {NUM_LAYERS}-layer LSTM with {HIDDEN_SIZE} hidden units")
print(f"   Sequence length: {SEQUENCE_LENGTH}")
print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Training epochs: {len(train_losses)}")

print("\n📈 PERFORMANCE METRICS:")
print(f"   RMSE: {results['rmse']:.2f}")
print(f"   MAE: {results['mae']:.2f}")
print(f"   Directional Accuracy: {results['directional_accuracy']:.2f}%")

print("\n🎯 COMPARISON WITH SINGLE-TIMEFRAME:")
print("   Previous results (single timeframe H4): ~52.19%")
print(f"   Current results (multi-timeframe): {results['directional_accuracy']:.2f}%")

improvement = results['directional_accuracy'] - 52.19
if improvement > 0:
    print(f"   ✅ Improvement: +{improvement:.2f}%")
elif improvement < 0:
    print(f"   ⚠️  Decrease: {improvement:.2f}%")
else:
    print("   ➖ No significant change")

print("\n💡 RECOMMENDATIONS:")

if results['directional_accuracy'] > 55:
    print("   ✅ Model shows promising results (>55% accuracy)")
    print("   Next steps:")
    print("   1. Run extensive backtesting with transaction costs")
    print("   2. Test on different market conditions (volatile/stable periods)")
    print("   3. Implement risk management (stop-loss, take-profit)")
    print("   4. Consider ensemble methods (combine multiple models)")
    print("   5. Paper trade before going live")
else:
    print("   ⚠️  Model needs improvement before trading")
    print("   Suggested improvements:")
    print("   1. Try different sequence lengths (30, 90, 120)")
    print("   2. Experiment with other architectures (GRU, Transformer, CNN-LSTM)")
    print("   3. Add more features (order flow, market sentiment, volume profile)")
    print("   4. Use attention mechanisms to focus on important timeframes")
    print("   5. Try other indices (Volatility 25, Crash 500 showed different patterns)")
    print("   6. Increase training data by combining multiple API calls")

print("\n🔬 ADVANCED TECHNIQUES TO TRY:")
print("   • Attention-based LSTM (focus on important features/timesteps)")
print("   • CNN-LSTM hybrid (extract local patterns + temporal dependencies)")
print("   • Transformer models (state-of-the-art for time series)")
print("   • Ensemble methods (combine LSTM + XGBoost + Prophet)")
print("   • Reinforcement Learning (learn optimal trading actions)")

print("\n" + "="*80)
print(" " * 25 + "ANALYSIS COMPLETE")
print("="*80)